# Week 8 — Block 2: Guided Demo (Image Data)

**DATS 6401 · Visualization of Complex Data**

~30 min on the sklearn digits (1,797 8×8 images — offline, instant):

1. Montage + class balance: the collection at a glance (~8 min)
2. Mean images + intensity histograms: structure (~8 min)
3. Channels on a synthetic RGB image (~4 min)
4. Occlusion saliency, from algorithm to overlay (~10 min)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
print(digits.images.shape, "| classes:", np.unique(digits.target))

## Part 1 — The collection at a glance

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(9, 4))
for ax, img, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap="gray"); ax.set_title(label, fontsize=8); ax.axis("off")
plt.suptitle("The montage: df.head() for images — scan for anomalies, out loud")
plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd
pd.Series(digits.target).value_counts().sort_index().plot.bar(
    figsize=(7, 2.4), color="#2E6E8E", rot=0, title="Class balance — check it BEFORE any accuracy number")
plt.show()

## Part 2 — Structure: prototypes & ink

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(9.5, 1.6))
for d, ax in enumerate(axes):
    ax.imshow(digits.images[digits.target == d].mean(axis=0), cmap="gray")
    ax.set_title(d, fontsize=9); ax.axis("off")
plt.suptitle("Mean images: blurry = varied; SIMILAR means (3 vs 8) forecast the confusion matrix")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 2.8))
for d, color in [(0, "#2E6E8E"), (1, "#d9534f")]:
    ax.hist(digits.images[digits.target == d].ravel(), bins=17, alpha=0.55,
            density=True, label=f"digit {d}", color=color)
ax.legend(); ax.set_title("Intensity distributions: a '1' uses far less ink than a '0'")
plt.show()

## Part 3 — Color = three grayscale stacks (synthetic, 60 seconds)

In [ ]:
H = W = 64
img = np.zeros((H, W, 3))
img[..., 0] = np.linspace(0, 1, W)[None, :]
yy, xx = np.mgrid[0:H, 0:W]
img[..., 1] = (((xx-32)**2 + (yy-32)**2) < 18**2).astype(float)
img[..., 2] = ((xx > 44) & (yy > 44)).astype(float)

fig, axes = plt.subplots(1, 4, figsize=(10, 2.6))
axes[0].imshow(img); axes[0].set_title("composite")
for k, (n, cm) in enumerate([("R", "Reds"), ("G", "Greens"), ("B", "Blues")]):
    axes[k+1].imshow(img[..., k], cmap=cm); axes[k+1].set_title(n)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## Part 4 — Occlusion saliency, built live

The algorithm on one slide of the deck; now each step is a line of code.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=2000).fit(digits.data, digits.target)   # 1 any classifier

img0 = digits.images[0].copy()
base = clf.predict_proba([img0.ravel()])[0][0]                            # 2 baseline P(true class)

sal = np.zeros((8, 8))
for i in range(8):                                                        # 3 slide the patch
    for j in range(8):
        t = img0.copy(); t[i, j] = 0
        sal[i, j] = base - clf.predict_proba([t.ravel()])[0][0]           # 4 damage done

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(img0, cmap="gray"); axes[0].set_title("a '0'")
axes[1].imshow(sal, cmap="hot"); axes[1].set_title("saliency: ΔP per hidden pixel")
axes[2].imshow(img0, cmap="gray"); axes[2].imshow(sal, cmap="hot", alpha=0.5)
axes[2].set_title("the deliverable: the OVERLAY")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

**Narrate the trust read:** the hot ring follows the stroke — the model keys on the digit, not the corners. Then the question for the room: *what would saliency on the background tell you, and who'd have to fix it?*

## Wrap-up → Block 3

Pipeline: **montage → balance → structure → saliency.** Block 3 reruns it and adds the image-browser app.